# Virtual Manipulation of GRIB2 with VirtualiZarr and grib2io

This notebook demonstrates how to use `VirtualiZarr` to open a grib2io kerchunk manifest
as a virtual dataset — holding only metadata references, no data bytes — and then
load actual data through the standard xarray backend interface (`engine="grib2io"`).


## Notebook Convention

Use the standard grib2io xarray backend interface directly.
Pass grib2io-specific options as keyword arguments to `xr.open_dataset(...)` or `xr.open_mfdataset(...)` (for example `use_icechunk`, `storage_options`, `filters`, `max_workers`, `network_timeout`, and `max_concurrent_requests`).

In [ ]:
import xarray as xr
import grib2io
import json
import pathlib
import os
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser
from grib2io.kerchunk import ReferenceGenerator

# Optional notebook-only dependency: do not require this in grib2io package deps.
try:
    from obstore.store import LocalStore
    from obspec_utils.registry import ObjectStoreRegistry

    OBSTORE_AVAILABLE = True
except ImportError:
    OBSTORE_AVAILABLE = False
    print("Optional packages not installed: obstore, obspec-utils. Install them in this notebook to enable registry-backed VirtualiZarr access.")

## 1. Build a Kerchunk Manifest

`ReferenceGenerator` scans a GRIB2 file and produces a kerchunk v1 reference manifest —
a plain dict mapping variable/coordinate names to byte-range references. No data is read.


In [1]:
_here = pathlib.Path(os.path.abspath(""))
_candidates = [
    _here / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
    _here.parent / "tests" / "input_data" / "gfs.t00z.pgrb2.1p00.f024",
]
grib2_file = next(str(p) for p in _candidates if p.exists())

gen = ReferenceGenerator(grib2_file)
manifest = gen.generate()
print(f"Manifest: {len(manifest['refs'])} refs for {grib2_file}")

# Persist for VirtualiZarr to read via URI
manifest_path = pathlib.Path(grib2_file).with_suffix(".json")
manifest_path.write_text(json.dumps(manifest))

NameError: name 'pathlib' is not defined

## 2. Open as a VirtualiZarr Dataset

`open_virtual_dataset` reads the manifest and returns an `xarray.Dataset` backed by
`ManifestArray` objects — virtual metadata only, no data downloaded. This is useful
for inspecting structure, combining datasets, or generating consolidated manifests.


In [ ]:
manifest_url = manifest_path.resolve().as_uri()

if OBSTORE_AVAILABLE:
    registry = ObjectStoreRegistry(
        {
            manifest_url: LocalStore(prefix=str(manifest_path.parent)),
        }
    )

    vds = open_virtual_dataset(
        url=manifest_url,
        registry=registry,
        parser=KerchunkJSONParser(),
        loadable_variables=[],
    )
else:
    # Fallback path keeps notebook runnable without obstore.
    vds = open_virtual_dataset(
        url=manifest_url,
        parser=KerchunkJSONParser(),
        loadable_variables=[],
    )

display(vds)

## 3. Load Actual Data

A `ManifestArray` is virtual, so calling `.compute()` on a VirtualiZarr dataset raises
an error. To load real data, use the standard grib2io/xarray interface:

- Open a manifest file with `xr.open_dataset(manifest_path, engine="grib2io")`
- Open a GRIB2 URL/path with `xr.open_dataset(grib2_file, engine="grib2io", use_icechunk=True)`

Notebook convention: pass grib2io-specific options directly as keyword arguments
to `xr.open_dataset(...)` / `xr.open_mfdataset(...)`.


In [ ]:
# Option A: manifest path -> xarray Dataset (through grib2io backend)
ds = xr.open_dataset(manifest_path, engine="grib2io")

# Option B: GRIB2 path/URL -> xarray Dataset via Icechunk-backed flow
# ds = xr.open_dataset(grib2_file, engine="grib2io", use_icechunk=True)

if "TMP" in ds.data_vars:
    data = ds.TMP.isel(y=slice(0, 10), x=slice(0, 10)).compute()
    print(f"TMP subset: shape={data.shape}, min={float(data.min()):.2f}, max={float(data.max()):.2f}")
    display(data)